In [0]:
# replace with your catalog
CATALOG = spark.catalog.currentCatalog()
CATALOG = "nikkthegreek"

In [0]:
import os
import sys
import platform
from lakehouse.spark import bronze, silver
from pyspark.sql import DataFrame, SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.testing import assertDataFrameEqual

In [0]:
if platform.system() == "Windows":
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
    print("Adding Python ENV variables on Windows")

In [0]:
try:
    spark
except NameError:
    builder = (
        SparkSession.builder.appName("Data with Nikk the Greek Spark Session")
        .master("local[4]")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config(
            "spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog",
        )
    )
    spark = configure_spark_with_delta_pip(builder).getOrCreate()

# 1. Set Up and Bronze Data

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

In [0]:
options = {"catalog": CATALOG, "target_schema": "bronze"}

In [0]:
class TestBronze(bronze.Bronze):
    def custom_load(self, table):
        df = spark.range(10).withColumn("t", F.lit(table))
        return df


bronze_instance = TestBronze(spark, **options)

In [0]:
bronze_instance.load().transform().write(mode="overwrite").execute("people", "planets")

In [0]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {df.count()}")
df.show()

In [0]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

# 2 Debug with Overwrite example

In [0]:
class TestSilver(silver.Silver):
    def custom_filter(self, df: DataFrame, table: str) -> DataFrame:
        return df.where("id <= 5")

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        return df.withColumn("col", F.lit("col"))


silver_instance = TestSilver(spark, **options)

## 2.1 Debug the data load

In [0]:
# Debug without filter
silver_instance.load().execute("people")
actual_df = silver_instance.data["people"]
expected_df = (
    spark.range(10)
    .withColumn("t", F.lit("people"))
    .withColumn("LH_BronzeTS", F.current_timestamp())
)
actual_df.show(truncate=False)
# assertSchemaEqual(actual_df, expected_df)
assertDataFrameEqual(actual_df.drop("LH_BronzeTS"), expected_df.drop("LH_BronzeTS"))

In [0]:
# Debug with filter
silver_instance.load(filter="custom").execute("people")
actual_df = silver_instance.data["people"]
expected_df = (
    spark.range(10)
    .withColumn("t", F.lit("people"))
    .withColumn("LH_BronzeTS", F.current_timestamp())
    .where("id <= 5")
)
actual_df.show(truncate=False)
# assertSchemaEqual(actual_df, expected_df)
assertDataFrameEqual(actual_df.drop("LH_BronzeTS"), expected_df.drop("LH_BronzeTS"))

# 2.2 Debug transformation

In [0]:
# Debug with default transformation
silver_instance.load(filter="custom").transform().execute("people")
actual_df = silver_instance.data["people"]
expected_df = (
    spark.range(10)
    .withColumn("t", F.lit("people"))
    .withColumn("LH_BronzeTS", F.current_timestamp())
    .where("id <= 5")
    .withColumn("col", F.lit("col"))
    .withColumn("LH_SilverTS", F.current_timestamp())
)
actual_df.show(truncate=False)
# assertSchemaEqual(actual_df, expected_df)
assertDataFrameEqual(
    actual_df.drop("LH_BronzeTS", "LH_SilverTS"),
    expected_df.drop("LH_BronzeTS", "LH_SilverTS"),
)

In [0]:
# Debug without default transformation
silver_instance.load(filter="custom").transform(ignore_defaults=True).execute("people")
actual_df = silver_instance.data["people"]
expected_df = (
    spark.range(10)
    .withColumn("t", F.lit("people"))
    .withColumn("LH_BronzeTS", F.current_timestamp())
    .where("id <= 5")
    .withColumn("col", F.lit("col"))
)
actual_df.show(truncate=False)
# assertSchemaEqual(actual_df, expected_df)
assertDataFrameEqual(actual_df.drop("LH_BronzeTS"), expected_df.drop("LH_BronzeTS"))

# 2.3 Debug write

In [0]:
silver_instance.load(filter="custom").transform().write(mode="overwrite").execute(
    "people"
)
actual_df = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
expected_df = (
    spark.range(10)
    .withColumn("t", F.lit("people"))
    .withColumn("LH_BronzeTS", F.current_timestamp())
    .where("id <= 5")
    .withColumn("col", F.lit("col"))
    .withColumn("LH_SilverTS", F.current_timestamp())
)
actual_df.show(truncate=False)
# assertSchemaEqual(actual_df, expected_df)
assertDataFrameEqual(
    actual_df.drop("LH_BronzeTS", "LH_SilverTS"),
    expected_df.drop("LH_BronzeTS", "LH_SilverTS"),
)

# 3 Clean Up

In [0]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.silver CASCADE")
#spark.stop()